# 📄 PDF Question Answering System

An NLP application that allows users to upload a PDF document and ask questions about its content using a Transformer-based Question Answering model.

## Project Overview

This project uses a pre-trained Transformer model to answer questions from PDF documents.

The application extracts text from a PDF, divides it into overlapping chunks, searches all chunks for the most relevant answer, and returns the answer with a confidence score.

## Import Libraries

Import all required libraries for PDF processing, NLP, timing, and data analysis.

In [ ]:
from transformers import AutoTokenizer, AutoModelForQuestionAnswering, pipeline
from tkinter import Tk, filedialog
from pypdf import PdfReader
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import pandas as pd
import time
import re

## Load Transformer Model

Load a pre-trained Transformer model and tokenizer for extractive Question Answering.

In [ ]:
model_name = "deepset/roberta-base-squad2"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForQuestionAnswering.from_pretrained(model_name)

print("Model loaded successfully!")

## Create Question Answering Pipeline

Create a Hugging Face pipeline that combines the tokenizer and model for question answering.

In [ ]:
qa_pipeline = pipeline(
    "question-answering",
    model=model,
    tokenizer=tokenizer
)

print("QA pipeline is ready!")

## Upload PDF

Select a PDF document from your computer and extract its text.

In [ ]:
root = Tk()
root.withdraw()

pdf_path = filedialog.askopenfilename(
    title="Select a PDF file",
    filetypes=[("PDF files", "*.pdf")]
)

reader = PdfReader(pdf_path)

context = ""

for page in reader.pages:
    text = page.extract_text()

    if text:
        context += text + "\n"

print("PDF loaded successfully!")
print("Pages:", len(reader.pages))

## Split PDF into Chunks

Long documents are divided into overlapping chunks to improve answer accuracy.

In [ ]:
def split_text(text, chunk_size=900, overlap=150):

    chunks = []

    start = 0

    while start < len(text):

        end = start + chunk_size

        chunks.append(text[start:end])

        start += chunk_size - overlap

    return chunks


chunks = split_text(context)

print("PDF loaded successfully!")
print(f"Pages: {len(reader.pages)}")
print(f"Chunks: {len(chunks)}")

## Ask Questions

Ask unlimited questions about the uploaded PDF. The system searches every chunk and returns the answer with the highest confidence.

In [ ]:
print("\nYou can now ask questions about the PDF.")
print("Type 'exit' to stop.\n")

import re
import time
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity


def clean_text(text):
    """Clean PDF text so retrieval and QA work better."""
    text = re.sub(r"\s+", " ", text)
    return text.strip()


chunks = [clean_text(chunk) for chunk in chunks if clean_text(chunk)]


def expand_question(question):
    """
    Add related words to improve retrieval.
    Example: 'What does Tala study?' may need words like major, degree, student.
    """
    q = question.lower()

    expansions = {
        "study": "study studies studying major degree program field university college student",
        "studies": "study studies studying major degree program field university college student",
        "gpa": "gpa grade grades score academic",
        "university": "university college school institution education",
        "where": "location place university college school",
        "skill": "skill skills technologies tools programming",
        "project": "project projects built developed created",
        "email": "email contact mail",
        "phone": "phone mobile number contact",
    }

    extra_words = []
    for word, related_words in expansions.items():
        if word in q:
            extra_words.append(related_words)

    return question + " " + " ".join(extra_words)


# Word + character TF-IDF makes retrieval more forgiving for PDF text.
word_vectorizer = TfidfVectorizer(
    stop_words="english",
    ngram_range=(1, 3),
    max_features=20000
)

char_vectorizer = TfidfVectorizer(
    analyzer="char_wb",
    ngram_range=(3, 5),
    max_features=20000
)

word_vectors = word_vectorizer.fit_transform(chunks)
char_vectors = char_vectorizer.fit_transform(chunks)


def retrieve_chunks(question, chunks, top_k=5, neighbor_window=1):
    """
    Retrieve strong chunks plus nearby chunks.
    Nearby chunks matter because PDFs often split one idea across multiple chunks.
    """
    expanded_question = expand_question(question)

    word_question_vector = word_vectorizer.transform([expanded_question])
    char_question_vector = char_vectorizer.transform([expanded_question])

    word_scores = cosine_similarity(word_question_vector, word_vectors).flatten()
    char_scores = cosine_similarity(char_question_vector, char_vectors).flatten()

    scores = (0.75 * word_scores) + (0.25 * char_scores)
    top_indices = scores.argsort()[::-1][:top_k]

    selected_indices = set()

    for index in top_indices:
        if scores[index] <= 0:
            continue

        start = max(0, index - neighbor_window)
        end = min(len(chunks), index + neighbor_window + 1)

        for nearby_index in range(start, end):
            selected_indices.add(nearby_index)

    return [chunks[index] for index in sorted(selected_indices)]


def answer_question(question, chunks):
    relevant_chunks = retrieve_chunks(
        question=question,
        chunks=chunks,
        top_k=5,
        neighbor_window=1
    )

    if not relevant_chunks:
        return "Answer not found clearly in the PDF.", 0.0

    best_result = None

    for chunk in relevant_chunks:
        result = qa_pipeline(
            question=question,
            context=chunk
        )

        if best_result is None or result["score"] > best_result["score"]:
            best_result = result

    answer = best_result["answer"].strip()
    confidence = float(best_result["score"])

    # DistilBERT scores can be low on CV/resume-style PDFs,
    # so do not reject useful short answers too aggressively.
    if answer == "":
        return "Answer not found clearly in the PDF.", confidence

    if confidence < 0.02 and len(answer.split()) <= 1:
        return "Answer not found clearly in the PDF.", confidence

    return answer, confidence


qa_history = []

while True:
    question = input("Ask a question: ").strip()

    if question.lower() == "exit":
        print("\nQuestion Answering ended.")
        break

    if question == "":
        print("Please enter a valid question.")
        continue

    start_time = time.time()

    answer, confidence = answer_question(question, chunks)

    inference_time = round(time.time() - start_time, 3)

    qa_history.append({
        "Question": question,
        "Answer": answer,
        "Confidence": round(confidence, 4),
        "Inference Time (s)": inference_time
    })

    print("\n" + "=" * 50)
    print(f"Question   : {question}")
    print(f"Answer     : {answer}")
    print(f"Confidence : {confidence:.2%}")
    print(f"Time       : {inference_time} sec")
    print("=" * 50)

## Display Results

Display all questions, answers, confidence scores, and inference times in a structured table.

In [ ]:
history_df = pd.DataFrame(qa_history)

history_df

## Conclusion

This project demonstrates how Transformer-based Question Answering models can be used to extract information from PDF documents.

To improve performance on long documents, the PDF is divided into overlapping chunks before inference.

Future work includes implementing Retrieval-Augmented Generation (RAG), semantic search, and a web interface.